In [3]:
import sys
from pathlib import Path
from torchvision import transforms,datasets
from torch.utils.data import Subset,DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score,precision_score,recall_score
sys.path.append(str(Path("..")))
from src.dataset import train_data,val_loader,train_dataset,test_path
from src.sampler import BalancedBatchSampler

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:

balanced_sampler = BalancedBatchSampler(
    train_dataset,
    batch_size=32,
    seed=42
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=balanced_sampler
)

num_classes = len(train_data.classes)

images, labels = next(iter(train_loader)) 
for class_id in range(num_classes):
    count = (labels == class_id).sum().item()
    print(train_data.classes[class_id], count)


ambulance 4
autobus 4
kamyun 4
kamyunet 4
minibus 4
savari 4
taxi 4
vanet 4


In [6]:

class CNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),

            nn.Linear(256, num_classes)
        )


    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


num_classes = len(train_data.classes)

model = CNN(num_classes)

model = model.to(device)


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


num_epochs = 100

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

val_f1_scores = []
val_precision_scores = []
val_recall_scores = []

val_f1_scores_by_class = []
val_precision_scores_by_class = []
val_recall_scores_by_class = []


for epoch in range(num_epochs):

    model.train()

    running_train_loss = 0.0

    train_correct = 0
    train_total = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()


        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)

        train_correct += (predicted == labels).sum().item()


    train_loss = running_train_loss / len(train_loader)

    train_accuracy = train_correct / train_total


    model.eval()

    running_val_loss = 0.0

    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()


            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()


            all_val_labels.extend(
                labels.cpu().numpy()
            )

            all_val_predictions.extend(
                predicted.cpu().numpy()
            )


    val_loss = running_val_loss / len(val_loader)

    val_accuracy = val_correct / val_total


    # =========================
    # Metrics
    # =========================

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    val_precision = precision_score(
        all_val_labels,
        all_val_predictions,
        average="macro",
        zero_division=0
    )

    val_recall = recall_score(
        all_val_labels,
        all_val_predictions,
        average="macro",
        zero_division=0
    )


    # Metrics for each class

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average=None,
        zero_division=0
    )

    precision_by_class = precision_score(
        all_val_labels,
        all_val_predictions,
        average=None,
        zero_division=0
    )

    recall_by_class = recall_score(
        all_val_labels,
        all_val_predictions,
        average=None,
        zero_division=0
    )


    # =========================
    # Save metrics
    # =========================

    train_losses.append(train_loss)

    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)

    val_accuracies.append(val_accuracy)

    val_f1_scores.append(val_f1)

    val_precision_scores.append(val_precision)

    val_recall_scores.append(val_recall)

    val_f1_scores_by_class.append(f1_score_by_class)

    val_precision_scores_by_class.append(precision_by_class)

    val_recall_scores_by_class.append(recall_by_class)


    # =========================
    # Print
    # =========================

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Precision: {val_precision:.4f} | "
        f"Val Recall: {val_recall:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )


    print("Precision by each class:")
    print(precision_by_class)

    print("Recall by each class:")
    print(recall_by_class)

    print("F1 Score by each class:")
    print(f1_score_by_class)

    print("=======================================================================================")





Epoch [1/100] | Train Loss: 1.1644 | Train Acc: 0.5870 | Val Loss: 0.8539 | Val Acc: 0.7102 | Val Precision: 0.7205 | Val Recall: 0.7000 | Val Macro F1: 0.6853
Precision by each class:
[0.64835165 0.83       0.94117647 0.52631579 0.6        0.80898876
 0.90909091 0.5       ]
Recall by each class:
[0.83098592 0.88297872 0.32323232 0.65934066 0.86746988 0.72
 0.81632653 0.5       ]
F1 Score by each class:
[0.72839506 0.8556701  0.48120301 0.58536585 0.70935961 0.76190476
 0.86021505 0.5       ]
Epoch [2/100] | Train Loss: 0.3887 | Train Acc: 0.8716 | Val Loss: 0.7330 | Val Acc: 0.7763 | Val Precision: 0.7646 | Val Recall: 0.7544 | Val Macro F1: 0.7558
Precision by each class:
[0.84482759 0.84536082 0.66666667 0.65656566 0.86111111 0.78301887
 1.         0.45945946]
Recall by each class:
[0.69014085 0.87234043 0.76767677 0.71428571 0.74698795 0.83
 0.84693878 0.56666667]
F1 Score by each class:
[0.75968992 0.85863874 0.71361502 0.68421053 0.8        0.80582524
 0.91712707 0.50746269]
Epoc

In [7]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.7679
Test Macro F1: 0.7604
Test F1 by class:
[0.8        0.91089109 0.61702128 0.62135922 0.80373832 0.81481481
 0.96969697 0.54545455]
